In [1]:
# ==========================================
# STAGE 1: DATA INGESTION & DATA CLEANING
# ==========================================

import numpy as np
import pandas as pd

# 1. Load your dataset from the project folder
df = pd.read_csv('readyflat.csv')

print('--- Raw Dataset Preview ---')
display(df.head(3))


# 2. Cleaning function for the Price column
# Converts strings like "32 L" or "1.14 Cr" into exact integer values in INR
def clean_price(val):
  if pd.isna(val):
    return np.nan
  val = str(val).strip().lower()
  try:
    if 'cr' in val:
      return float(val.replace('cr', '').strip()) * 10000000
    elif 'l' in val:
      return float(val.replace('l', '').strip()) * 100000
    else:
      return float(val)
  except:
    return np.nan


df['Cleaned_Price'] = df['Price'].apply(clean_price)

# 3. Feature Extraction for Room Count (Bedrooms)
# Extracts the number before "BHK" from the Apartment column
df['Bedrooms'] = (
    df['Apartment']
    .str.extract(r'(\d+)\s*BHK', expand=False)
    .astype(float)
    .fillna(2)
)

# 4. Filter columns needed for our Linear Regression model (Square Footage & Room Count)
model_data = df[['SqftArea', 'Bedrooms', 'Cleaned_Price']].dropna()

print(f'\nCleaned Data Shape for Modeling: {model_data.shape}')
display(model_data.head(3))

--- Raw Dataset Preview ---


,Unnamed: 0.1,Unnamed: 0,Apartment,Society,Location,Price,Price/sqft,SqftArea,Construction_Status,Year's Old,bathrooms,type of property
0,0,0,2 BHK Apartment,NaN,Baruipur,32 L,"3,265 / sq ft",980,Ready to move,NaN,2 Bathrooms,Resale
1,1,1,2 BHK Apartment,NaN,Garia,36 L,"4,645 / sq ft",775,Ready to move,NaN,2 Bathrooms,Resale
2,2,2,2 BHK Independent Floor,NaN,Baranagar,24 L,"3,243 / sq ft",740,Ready to move,NaN,1 Bathrooms,Resale



Cleaned Data Shape for Modeling: (400, 3)


,SqftArea,Bedrooms,Cleaned_Price
0,980,2.0,3200000.0
1,775,2.0,3600000.0
2,740,2.0,2400000.0


In [2]:
# ==========================================
# STAGE 2: TRAIN-TEST SPLIT & MODEL TRAINING
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Define Features (X) and Target (y)
X = model_data[['SqftArea', 'Bedrooms']]
y = model_data['Cleaned_Price']

# 2. Split dataset: 80% training, 20% testing
# random_state ensures reproducibility so your split is identical every time you run it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape : {X_test.shape}")

# 3. Initialize and Train the Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 4. Display the Learned Model Parameters
print("\n--- Model Parameters Learned ---")
print(f"Intercept (Base Price) : ₹{lr_model.intercept_:,.2f}")
print(f"Coefficients (Weights) : SquareFeet = {lr_model.coef_[0]:.2f}, Bedrooms = {lr_model.coef_[1]:.2f}")

Training Features Shape: (320, 2)
Testing Features Shape : (80, 2)

--- Model Parameters Learned ---
Intercept (Base Price) : ₹-3,689,986.92
Coefficients (Weights) : SquareFeet = 7082.63, Bedrooms = 671536.72


In [3]:
# ==========================================
# STAGE 3: EVALUATION & PREDICTION
# ==========================================

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Generate predictions on the test set (unseen data)
y_pred = lr_model.predict(X_test)

# 2. Calculate Evaluation Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- Model Evaluation Metrics ---")
print(f"R² Score : {r2:.4f}")
print(f"MAE      : ₹{mae:,.2f}")
print(f"RMSE     : ₹{rmse:,.2f}")

# 3. Predict price for a brand new property in West Bengal
# Example: 1,200 sq.ft. apartment with 3 bedrooms
new_property = pd.DataFrame({'SqftArea': [1200], 'Bedrooms': [3]})

predicted_price = lr_model.predict(new_property)
print('\n--- Prediction for New Property ---')
print(f'Property Details: 1,200 Sq.Ft | 3 BHK')
print(f'Estimated Price : ₹{predicted_price[0]:,.2f}')

--- Model Evaluation Metrics ---
R² Score : 0.4498
MAE      : ₹2,482,042.05
RMSE     : ₹4,761,777.13

--- Prediction for New Property ---
Property Details: 1,200 Sq.Ft | 3 BHK
Estimated Price : ₹6,823,782.29


In [5]:
import os
import joblib

# 1. Create the 'models' directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# 2. Save the trained model pipeline/object
joblib.dump(lr_model, 'models/house_price_model.pkl')

print("Model successfully saved to 'models/house_price_model.pkl'!")

Model successfully saved to 'models/house_price_model.pkl'!


In [6]:
# ==========================================
# UPGRADED PIPELINE: ADDING LOCATION FEATURE
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. Prepare features including Location
model_data = df[['SqftArea', 'Bedrooms', 'Location', 'Cleaned_Price']].dropna()

X = model_data[['SqftArea', 'Bedrooms', 'Location']]
y = model_data['Cleaned_Price']

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Create Preprocessor for Categorical Location Data
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Location'])
    ],
    remainder='passthrough'
)

# 4. Build Modeling Pipeline
upgraded_pipeline = Pipeline(
    steps=[('preprocessor', preprocessor), ('model', LinearRegression())]
)

# 5. Train and Evaluate
upgraded_pipeline.fit(X_train, y_train)
y_pred = upgraded_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('--- Upgraded Model Evaluation Metrics ---')
print(f'R² Score : {r2:.4f}')
print(f'MAE      : ₹{mae:,.2f}')
print(f'RMSE     : ₹{rmse:,.2f}')

# 6. Test Prediction with Location
new_property_wb = pd.DataFrame(
    {'SqftArea': [1200], 'Bedrooms': [3], 'Location': ['New Town']}
)
predicted_price_wb = upgraded_pipeline.predict(new_property_wb)
print('\n--- Prediction for New West Bengal Property ---')
print(f'Property Details: 1,200 Sq.Ft | 3 BHK | New Town')
print(f'Estimated Price : ₹{predicted_price_wb[0]:,.2f}')

--- Upgraded Model Evaluation Metrics ---
R² Score : 0.6966
MAE      : ₹1,545,434.75
RMSE     : ₹3,535,895.55

--- Prediction for New West Bengal Property ---
Property Details: 1,200 Sq.Ft | 3 BHK | New Town
Estimated Price : ₹6,540,800.86


In [7]:
import os
import joblib

# Create 'models' folder if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save our UPGRADED pipeline
joblib.dump(upgraded_pipeline, 'models/house_price_model.pkl')
print(
    "Upgraded model successfully saved as 'models/house_price_model.pkl'!"
)

Upgraded model successfully saved as 'models/house_price_model.pkl'!


In [8]:
# ==========================================
# SCRIPT TO GENERATE app.py AUTOMATICALLY
# ==========================================

app_code = """import streamlit as st
import pandas as pd
import joblib

# Load the saved upgraded model pipeline
model = joblib.load('models/house_price_model.pkl')

st.title("🏠 West Bengal Real Estate Price Predictor")
st.write("Interactive ML Web App built with scikit-learn and Streamlit.")

# Sidebar or main inputs for property features
st.header("Enter Property Details:")

sqft = st.number_input("Square Feet Area (SqftArea)", min_value=200, max_value=10000, value=1200)
bedrooms = st.selectbox("Number of Bedrooms (BHK)", [1, 2, 3, 4, 5])

# List of common locations from your dataset or general West Bengal regions
location_options = [
    'New Town', 'Salt Lake', 'Garia', 'Park Street', 'Behala', 
    'Howrah', 'Jadavpur', 'Tollygunge', 'Baruipur', 'Baranagar', 'Gariahat'
]
location = st.selectbox("Location", location_options)

# Prediction button
if st.button("Predict Property Price"):
    # Create a dataframe for the model input
    input_df = pd.DataFrame({
        'SqftArea': [sqft],
        'Bedrooms': [bedrooms],
        'Location': [location]
    })
    
    # Make prediction using our upgraded pipeline
    prediction = model.predict(input_df)[0]
    
    # Display the result clearly
    st.success(f"Estimated Market Price: ₹ {prediction:,.2f} (~{prediction / 100000:.2f} Lakhs)")
"""

# Write the code into a file named app.py
with open("app.py", "w") as f:
    f.write(app_code)

print("Successfully created 'app.py' in your project directory!")

Successfully created 'app.py' in your project directory!
